# 05 — Chroma: industry-wise schema + Word/PDF upload with LangChain

Everything is in this notebook: schema, embedding, LangChain loading, ingestion gates, Chroma store, tests.

| # | Section |
|---|---|
| 0 | Setup (uv `.venv` kernel) |
| 1 | Schema: base record + 8 industry models (Pydantic = single source of truth) |
| 2 | Embedding model (pinned, cached by content hash) |
| 3 | Load Word/PDF with **LangChain** + attach planned metadata from the manifest |
| 4 | Ingestion gates: source → purpose → classification → ACL → retention → sensitive scan → section chunking → validate |
| 5 | Chroma store (embedded `PersistentClient`): one cosine collection per industry, typed metadata, `CURRENT` alias dict, SQLite system of record |
| 6 | Upload all industry documents (bad docs → quarantine) |
| 7 | Idempotency · 8 Staging → promote · 9 Safe search · 10 Isolation · 11 Update · 12 Delete · 13 Metrics |

Inputs: `data/docs/<industry>/*.pdf|*.docx` + `manifest.jsonl` (made by `scripts/make_docs.py`, all content fake).
Sections 0-4 are identical to `01_pgvector.ipynb` (same schema, same chunks); only the store and the tests change.

**Isolation in Chroma is filter-only**: embedded Chroma has no row-level security and no per-request auth. Tenant + ACL are enforced only because every read goes through `safe_search`, which puts them in the `where` pre-filter.
Design: [docs/PLAN.md](../docs/PLAN.md) · Setup: [SETUP.md](../SETUP.md)

## 0. Setup

In [ ]:
import os, re, json, math, time, hashlib, statistics, types, warnings
from datetime import date, datetime, timezone
from pathlib import Path
from typing import ClassVar, Literal, Union, get_args, get_origin
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # model already cached locally; delete this line on a new machine
DOCS, SAMPLES, RESULTS = ROOT / "data" / "docs", ROOT / "data" / "samples", ROOT / "data" / "results"
pd.set_option("display.max_colwidth", 90)

import sys; print("python:", sys.executable)

## 1. Schema — one record = one retrievable chunk

Base groups from the blueprint: **identity, provenance, content/vector, access/context, lifecycle**. Each industry adds its own typed fields and rules. Validation is the last gate: anything invalid never reaches the index.

In [ ]:
SCHEMA_VERSION = 1
TOKEN = r"^tok_[a-z0-9]+$"          # tokenized identifier, never a raw ID
Status = Literal["staged", "active", "superseded", "deleted"]
Classification = Literal["public", "internal", "confidential", "restricted"]


def make_chunk_id(tenant_id, source_id, source_version, chunk_no) -> str:
    """Deterministic -> idempotent upserts."""
    return hashlib.sha256(f"{tenant_id}|{source_id}|{source_version}|{chunk_no}".encode()).hexdigest()


def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


class BaseChunk(BaseModel):
    model_config = ConfigDict(extra="forbid")
    INDUSTRY: ClassVar[str] = ""
    # identity
    chunk_id: str
    tenant_id: str = Field(min_length=1)
    industry: str
    entity_type: str
    # provenance
    source_id: str
    source_version: str
    source_file: str
    chunk_no: int = Field(ge=0)
    section: str | None = None
    content_hash: str
    parent_id: str | None = None
    # content / vector
    text: str = Field(min_length=1)
    embedding: list[float]
    embedding_model: str
    language: str = "en"
    # access / context
    classification: Classification
    allowed_groups: list[str] = Field(min_length=1)        # fail closed: no ACL -> reject
    jurisdiction: str
    valid_from: date
    valid_to: date | None = None
    # lifecycle
    retention_until: date
    status: Status = "staged"
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

    @model_validator(mode="after")
    def _base_guardrails(self):
        if self.chunk_id != make_chunk_id(self.tenant_id, self.source_id, self.source_version, self.chunk_no):
            raise ValueError("chunk_id is not the stable hash of tenant|source|version|chunk_no")
        if self.content_hash != content_hash(self.text):
            raise ValueError("content_hash does not match text")
        if not self.embedding or not all(math.isfinite(x) for x in self.embedding):
            raise ValueError("embedding empty or non-finite")
        if self.valid_to and self.valid_to < self.valid_from:
            raise ValueError("valid_to before valid_from")
        if self.retention_until < date.today():
            raise ValueError("retention_until already passed")
        return self


BASE_FIELDS = frozenset(BaseChunk.model_fields)

In [ ]:
class HealthcareChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "healthcare"
    industry: Literal["healthcare"] = "healthcare"
    patient_ref_token: str = Field(pattern=TOKEN)
    record_type: str
    provider: str
    encounter_date: date
    consent_scope: Literal["treatment", "care_coordination", "research"]


class BankingChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "banking"
    industry: Literal["banking"] = "banking"
    customer_ref_token: str | None = Field(default=None, pattern=TOKEN)
    product: str
    account_type: str
    document_type: str
    effective_date: date


class InsuranceChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "insurance"
    industry: Literal["insurance"] = "insurance"
    policy_ref: str = Field(pattern=TOKEN)
    claim_ref: str | None = Field(default=None, pattern=TOKEN)
    line_of_business: str
    event_date: date | None = None
    claim_status: Literal["open", "closed", "n/a"]


class LegalChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "legal"
    industry: Literal["legal"] = "legal"
    matter_id: str
    client_ref: str = Field(pattern=TOKEN)
    privilege_status: Literal["none", "privileged", "work_product"]
    document_type: str
    effective_date: date

    @model_validator(mode="after")
    def _privilege(self):
        if self.privilege_status != "none" and self.classification != "restricted":
            raise ValueError("privileged/work-product content must be classified 'restricted'")
        return self


class RetailChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "retail"
    industry: Literal["retail"] = "retail"
    product_id: str
    sku: str
    brand: str
    locale: str
    channel: Literal["online", "store", "omni"]
    price_valid_from: date | None = None
    price_valid_to: date | None = None
    inventory_region: str


class ManufacturingChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "manufacturing"
    industry: Literal["manufacturing"] = "manufacturing"
    asset_id: str
    site_id: str
    equipment_type: str
    safety_class: Literal["low", "medium", "high"]
    revision: str
    document_owner: str
    approval_status: Literal["approved"]          # draft/obsolete procedures are never indexed


class GovernmentChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "government"
    industry: Literal["government"] = "government"
    agency_id: str
    record_class: str
    release_status: Literal["public_release", "restricted_release"]
    case_ref_token: str | None = Field(default=None, pattern=TOKEN)

    @model_validator(mode="after")
    def _release(self):
        if self.classification == "public" and self.release_status != "public_release":
            raise ValueError("public classification requires release_status=public_release")
        return self


class EducationHrChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "education_hr"
    industry: Literal["education_hr"] = "education_hr"
    domain: Literal["education", "hr"]
    org_unit: str
    course_or_department: str
    term: str | None = None
    region: str
    subject_ref_token: str | None = Field(default=None, pattern=TOKEN)


INDUSTRIES: dict[str, type[BaseChunk]] = {m.INDUSTRY: m for m in (
    HealthcareChunk, BankingChunk, InsuranceChunk, LegalChunk,
    RetailChunk, ManufacturingChunk, GovernmentChunk, EducationHrChunk)}

print("base fields:", ", ".join(BaseChunk.model_fields))
pd.DataFrame([{"industry": i, "model": m.__name__,
               "industry fields": ", ".join(f for f in m.model_fields if f not in BASE_FIELDS)}
              for i, m in INDUSTRIES.items()])

## 2. Embedding model — pinned, normalized, cached by content hash

In [ ]:
from sentence_transformers import SentenceTransformer


class Embedder:
    QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge v1.5 query instruction

    def __init__(self, model_name="BAAI/bge-small-en-v1.5", tag="bgesmall"):
        self.model_name, self.tag = model_name, tag
        self.model = SentenceTransformer(model_name)
        self.dim = (getattr(self.model, "get_embedding_dimension", None) or self.model.get_sentence_embedding_dimension)()
        self._cache, self.cache_hits = {}, 0

    def embed(self, texts):
        keys = [content_hash(t) for t in texts]
        todo = [(k, t) for k, t in zip(keys, texts) if k not in self._cache]
        self.cache_hits += len(texts) - len(todo)
        if todo:
            for (k, _), v in zip(todo, self.model.encode([t for _, t in todo], normalize_embeddings=True)):
                self._cache[k] = v.astype(float).tolist()
        return [self._cache[k] for k in keys]

    def embed_query(self, q):
        return self.model.encode(self.QUERY_PREFIX + q, normalize_embeddings=True).astype(float).tolist()


embedder = Embedder()
print(embedder.model_name, "| dim", embedder.dim)

## 3. Load Word / PDF with LangChain + attach the planned metadata

- `PyPDFLoader` (one `Document` per page) and `Docx2txtLoader` (one `Document` per file).
- `manifest.jsonl` is the source system's metadata export: tenant, ACL, classification, jurisdiction, validity, retention, industry fields. It is merged into `Document.metadata`.
- A file **without** a manifest entry is rejected: no metadata = no ACL = fail closed.
- **Clean** step: drop repeated header/footer lines (banner, address/footer bars, signature line) so they never get embedded.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)          # langchain-community sunset notice
    from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

LOADERS = {".pdf": PyPDFLoader, ".docx": Docx2txtLoader}
CLASS_WORDS = {"PUBLIC", "INTERNAL", "CONFIDENTIAL", "RESTRICTED"}


@dataclass
class Rejection:
    industry: str
    source_id: str
    gate: str
    reason: str


def clean_text(text: str) -> str:
    """Blueprint 'CLEAN' step: remove page furniture, keep content."""
    keep = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s in CLASS_WORDS:                       # empty / classification banner
            continue
        if "  |  " in s or s.startswith("Approved by:"):    # address bar, footer bar, signature line
            continue
        keep.append(re.sub(r"\s{2,}", " ", s))
    return "\n".join(keep)


def load_langchain_documents(industry: str):
    folder = DOCS / industry
    manifest = {m["file"]: m for m in (json.loads(l) for l in (folder / "manifest.jsonl").read_text("utf-8").splitlines() if l)}
    docs, rejections = [], []
    for path in sorted(p for p in folder.iterdir() if p.suffix.lower() in LOADERS):
        meta = manifest.get(path.name)
        if meta is None:
            rejections.append(Rejection(industry, path.stem, "metadata", f"{path.name} has no manifest entry")); continue
        loader = LOADERS[path.suffix.lower()]
        for page in loader(str(path)).load():
            page.metadata = {**page.metadata, **meta, "loader": loader.__name__}
            page.page_content = clean_text(page.page_content)
            docs.append(page)
    return docs, rejections


def to_source_docs(lc_docs: list[Document]) -> list[dict]:
    """Merge pages of the same file into one source doc (pipeline input)."""
    merged = {}
    for d in lc_docs:
        m = merged.setdefault(d.metadata["file"], {**{k: v for k, v in d.metadata.items()
                                                       if k not in ("page", "page_label", "total_pages")}, "text": ""})
        m["text"] = (m["text"] + "\n" + d.page_content).strip()
    return list(merged.values())


lc_docs, _ = load_langchain_documents("healthcare")
d0 = lc_docs[0]
print("page_content (cleaned):\n", d0.page_content[:600], "...\n")
{k: d0.metadata[k] for k in ["source", "loader", "file", "source_id", "tenant_id", "classification",
                             "allowed_groups", "jurisdiction", "valid_from", "retention_until", "metadata"]}

## 4. Ingestion gates (fail closed)

Order: **authorized source → approved purpose → classified → ACL present → retention valid → sensitive scan (reject secrets/SSN/cards, redact email/phone) → section-aware chunking (LangChain splitter) → embed → schema validation**. Each chunk text is prefixed with `title - section` so it stays meaningful on its own.

In [ ]:
APPROVED_SOURCES = {"ehr_export", "core_banking", "claims_system", "dms_legal", "pim", "plm", "records_mgmt", "lms_hris"}
APPROVED_PURPOSES = {"rag_search", "support_assistant"}
SECRETS = [re.compile(p) for p in (r"AKIA[0-9A-Z]{16}", r"-----BEGIN [A-Z ]*PRIVATE KEY-----",
                                   r"(?i)\b(password|passwd|api[_-]?key|secret)\s*[:=]\s*\S+")]
SSN = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
CARD = re.compile(r"\b(?:\d[ -]?){13,19}\b")
EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b")
PHONE = re.compile(r"\+?\d{1,2}[ .-]?\(?\d{3}\)?[ .-]?\d{3}[ .-]?\d{4}\b")
SECTION = re.compile(r"^(\d+\. [A-Z][A-Za-z ]+|Revision History)$", re.M)

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40,
                                          separators=[". ", "; ", " ", ""], keep_separator="end")


def luhn(digits: str) -> bool:
    nums = [int(x) for x in digits][::-1]
    return sum(n if i % 2 == 0 else (n * 2 - 9 if n * 2 > 9 else n * 2) for i, n in enumerate(nums)) % 10 == 0


def sensitive_scan(text):
    """(reject_reason | None, redacted_text)"""
    if any(p.search(text) for p in SECRETS):
        return "credential/secret detected", text
    if SSN.search(text):
        return "SSN detected", text
    for m in CARD.finditer(text):
        digits = re.sub(r"\D", "", m.group())
        if 13 <= len(digits) <= 19 and luhn(digits):
            return "payment card number detected", text
    return None, PHONE.sub("[PHONE]", EMAIL.sub("[EMAIL]", text))


def split_sections(text, title):
    """[(section, body)] using the document's numbered headings; text before the first heading = control block."""
    marks = list(SECTION.finditer(text))
    parts = [("Document control", text[:marks[0].start()] if marks else text)]
    for i, m in enumerate(marks):
        parts.append((m.group(1), text[m.end(): marks[i + 1].start() if i + 1 < len(marks) else len(text)]))
    return [(s, " ".join(b.replace(title, "", 1).split() if s == "Document control" else b.split()))
            for s, b in parts if b.strip()]


@dataclass
class IngestResult:
    records: list = field(default_factory=list)
    rejections: list = field(default_factory=list)
    clean_docs: list = field(default_factory=list)


def run_pipeline(docs, industry, embedder) -> IngestResult:
    model, out = INDUSTRIES[industry], IngestResult()
    for doc in docs:
        sid = doc.get("source_id", "?")
        reject = lambda gate, why: out.rejections.append(Rejection(industry, sid, gate, why))
        if doc.get("source_system") not in APPROVED_SOURCES:
            reject("authorized_source", f"source_system '{doc.get('source_system')}' not approved"); continue
        if doc.get("purpose") not in APPROVED_PURPOSES:
            reject("approved_purpose", f"purpose '{doc.get('purpose')}' not approved"); continue
        if not doc.get("classification"):
            reject("classified", "missing classification"); continue
        if not doc.get("allowed_groups"):
            reject("acl", "no allowed_groups (ACL)"); continue
        if date.fromisoformat(doc["retention_until"]) < date.today():
            reject("retention", f"retention_until {doc['retention_until']} already passed"); continue
        why, text = sensitive_scan(doc["text"])
        if why:
            reject("sensitive_scan", why); continue

        pieces = [(sec, f"{doc['title']} - {sec}: {p}") for sec, body in split_sections(text, doc["title"])
                  for p in splitter.split_text(body)]
        vecs = embedder.embed([p for _, p in pieces])
        try:
            built = [model(
                chunk_id=make_chunk_id(doc["tenant_id"], sid, doc["source_version"], i),
                tenant_id=doc["tenant_id"], entity_type=doc["entity_type"], source_id=sid,
                source_version=doc["source_version"], source_file=doc["file"], chunk_no=i, section=sec,
                content_hash=content_hash(p), parent_id=sid, text=p, embedding=v,
                embedding_model=embedder.model_name, language=doc.get("language", "en"),
                classification=doc["classification"], allowed_groups=doc["allowed_groups"],
                jurisdiction=doc["jurisdiction"], valid_from=doc["valid_from"], valid_to=doc.get("valid_to"),
                retention_until=doc["retention_until"], **doc.get("metadata", {}))
                for i, ((sec, p), v) in enumerate(zip(pieces, vecs))]
        except ValidationError as e:
            err = e.errors()[0]
            reject("schema_validation", f"{'.'.join(map(str, err['loc'])) or 'record'}: {err['msg']}"); continue
        out.records += built
        out.clean_docs.append({**doc, "text": text})
    return out


demo = run_pipeline(to_source_docs(lc_docs), "healthcare", embedder)
pd.DataFrame([{"source_id": r.source_id, "chunk_no": r.chunk_no, "section": r.section, "text": r.text}
              for r in demo.records if r.source_id == "hc-001"])

## 5. Chroma store — one collection per industry

- Embedded `chromadb.PersistentClient(path="data/chroma")`. Collection per industry `{industry}_v1_bgesmall_384`, HNSW **cosine** space, `embedding_function=None` (we always pass our own vectors).
- Chroma has **no alias**: a `CURRENT` dict `industry → collection name` is the blue/green switch point; every function resolves through it.
- Metadata is generated from the Pydantic model: `allowed_groups` stays a **list** (this Chroma version stores list metadata and supports `$contains` on it, tested below); dates become `int yyyymmdd` so `$lte/$gte` range-compare; `valid_to = null` → `99991231` sentinel; other `None` values are omitted (Chroma rejects `None`).
- Id = `chunk_id` (deterministic sha256) → `upsert` is idempotent. Upsert keeps an existing chunk's `status` (Chroma `upsert` would otherwise overwrite it).
- `safe_search` puts tenant + ACL + `status == active` + validity window in the `where` clause of `collection.query`; Chroma applies the metadata filter **before** the ANN scan, so top-k is never post-filtered in Python.
- System of record = per-notebook SQLite `data/sor/chroma.db` (`sources`, `quarantine`), redacted text only.

In [ ]:
import sqlite3
import chromadb
from chromadb.config import Settings

CHROMA_PATH, SOR_PATH = ROOT / "data" / "chroma", ROOT / "data" / "sor" / "chroma.db"
VALID_TO_OPEN = 99991231                  # sentinel for valid_to = null
LIST_FIELDS = {"allowed_groups"}


def index_name(industry):
    return f"{industry}_v{SCHEMA_VERSION}_{embedder.tag}_{embedder.dim}"


def ymd(d) -> int:
    d = date.fromisoformat(d) if isinstance(d, str) else d
    return d.year * 10000 + d.month * 100 + d.day


def today_ymd() -> int:
    return ymd(date.today())


def to_metadata(rec: BaseChunk) -> dict:
    """Pydantic record -> Chroma metadata (scalars + str list; dates as int yyyymmdd; None omitted)."""
    meta = {}
    for k, v in rec.model_dump(exclude={"text", "embedding"}).items():
        if k == "valid_to" and v is None:
            v = VALID_TO_OPEN
        if v is None:
            continue
        if isinstance(v, datetime):
            v = v.isoformat()
        elif isinstance(v, date):
            v = ymd(v)
        meta[k] = v
    return meta


# --- system of record (SQLite, stdlib) ---------------------------------------
SOR_DDL = """
CREATE TABLE IF NOT EXISTS sources (
  source_id TEXT NOT NULL, source_version TEXT NOT NULL, tenant_id TEXT NOT NULL, industry TEXT NOT NULL,
  title TEXT, source_file TEXT, text TEXT NOT NULL, raw JSON NOT NULL, retention_until TEXT NOT NULL,
  ingested_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP, PRIMARY KEY (source_id, source_version));
CREATE TABLE IF NOT EXISTS quarantine (
  id INTEGER PRIMARY KEY AUTOINCREMENT, industry TEXT, source_id TEXT, gate TEXT, reason TEXT,
  at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP, UNIQUE (industry, source_id, gate, reason));
"""


def connect_sor():
    SOR_PATH.parent.mkdir(parents=True, exist_ok=True)
    db = sqlite3.connect(SOR_PATH, isolation_level=None)      # autocommit
    db.executescript(SOR_DDL)
    return db


def save_sources(industry, docs):
    sor.executemany(
        "INSERT INTO sources (source_id, source_version, tenant_id, industry, title, source_file, text, raw, retention_until) "
        "VALUES (?,?,?,?,?,?,?,?,?) ON CONFLICT (source_id, source_version) DO UPDATE "
        "SET text = excluded.text, raw = excluded.raw, retention_until = excluded.retention_until",
        [(d["source_id"], d["source_version"], d["tenant_id"], industry, d.get("title"), d.get("file"), d["text"],
          json.dumps({k: v for k, v in d.items() if k != "text"}), d["retention_until"]) for d in docs])


def save_rejections(rejections):
    sor.executemany("INSERT OR IGNORE INTO quarantine (industry, source_id, gate, reason) VALUES (?,?,?,?)",
                    [(r.industry, r.source_id, r.gate, r.reason) for r in rejections])


# --- Chroma store --------------------------------------------------------------
CURRENT: dict[str, str] = {}              # alias: industry -> live collection (Chroma has no native alias)


def col(industry):
    return client.get_collection(CURRENT[industry])


def create_indexes(recreate=True):
    """One cosine collection per industry. recreate=True drops our collections first -> clean re-runs."""
    existing = {c.name for c in client.list_collections()}
    if recreate:                          # SoR is rebuilt together with the index -> no rows left over from a previous run
        sor.execute("DELETE FROM sources")
        sor.execute("DELETE FROM quarantine")
    for ind in INDUSTRIES:
        name = index_name(ind)
        if recreate and name in existing:
            client.delete_collection(name)
        client.get_or_create_collection(
            name, embedding_function=None, configuration={"hnsw": {"space": "cosine"}},
            metadata={"industry": ind, "schema_version": SCHEMA_VERSION, "embedding_model": embedder.model_name,
                      "dim": embedder.dim})
        CURRENT[ind] = name
    return dict(CURRENT)


def upsert(industry, records):
    if not records:
        return 0
    c = col(industry)
    ids = [r.chunk_id for r in records]
    prev = c.get(ids=ids, include=["metadatas"])
    keep = {i: m["status"] for i, m in zip(prev["ids"], prev["metadatas"])}   # never reset lifecycle on re-upload
    metas = [to_metadata(r) | ({"status": keep[r.chunk_id]} if r.chunk_id in keep else {}) for r in records]
    for s in range(0, len(ids), client.get_max_batch_size()):
        e = s + client.get_max_batch_size()
        c.upsert(ids=ids[s:e], embeddings=[r.embedding for r in records[s:e]],
                 documents=[r.text for r in records[s:e]], metadatas=metas[s:e])
    return len(ids)


def _set_status(industry, where, status):
    c = col(industry)
    ids = c.get(where=where, include=[])["ids"]
    if ids:
        c.update(ids=ids, metadatas=[{"status": status}] * len(ids))     # update merges metadata keys
    return len(ids)


def promote(industry, source_ids=None):
    where = {"status": "staged"} if not source_ids else {"$and": [{"status": "staged"}, {"source_id": {"$in": source_ids}}]}
    return _set_status(industry, where, "active")


def supersede(industry, source_id, keep_version):
    return _set_status(industry, {"$and": [{"source_id": source_id}, {"source_version": {"$ne": keep_version}},
                                           {"status": "active"}]}, "superseded")


def purge(industry, source_id):
    c = col(industry)
    n = len(c.get(where={"source_id": source_id}, include=[])["ids"])
    c.delete(where={"source_id": source_id})
    sor.execute("DELETE FROM sources WHERE source_id = ?", (source_id,))
    return n


def count(industry):
    return col(industry).count()


def any_of(clauses):
    return clauses[0] if len(clauses) == 1 else {"$or": clauses}       # $or needs >= 2 operands


def safe_where(tenant_id, groups):
    t = today_ymd()
    return {"$and": [{"tenant_id": tenant_id},
                     any_of([{"allowed_groups": {"$contains": g}} for g in groups]),
                     {"status": "active"}, {"valid_from": {"$lte": t}}, {"valid_to": {"$gte": t}}]}


def safe_search(industry, tenant_id, groups, query_vec, k=5):
    """Tenant + ACL (any-match) + status + validity are a PRE-FILTER inside collection.query (no Python post-filter)."""
    if not tenant_id or not groups or not all(isinstance(g, str) and g for g in groups):
        raise PermissionError("safe_search needs tenant_id and at least one group")
    r = col(industry).query(query_embeddings=[list(query_vec)], n_results=k, where=safe_where(tenant_id, groups),
                            include=["metadatas", "documents", "distances"])
    return [{"chunk_id": i, "tenant_id": m["tenant_id"], "source_id": m["source_id"], "source_version": m["source_version"],
             "section": m.get("section"), "text": d, "score": 1 - dist}
            for i, m, d, dist in zip(r["ids"][0], r["metadatas"][0], r["documents"][0], r["distances"][0])]


client = chromadb.PersistentClient(path=str(CHROMA_PATH), settings=Settings(anonymized_telemetry=False))   # no phone-home
sor = connect_sor()
print("chromadb", chromadb.__version__, "| path", CHROMA_PATH, "| SoR", SOR_PATH)
print("sample metadata for one healthcare chunk:")
to_metadata(demo.records[0])

In [ ]:
tables = create_indexes()

# capability check this design relies on: list metadata + $contains on a list field
probe = client.get_or_create_collection("capability_probe", embedding_function=None, configuration={"hnsw": {"space": "cosine"}})
probe.upsert(ids=["p1", "p2"], embeddings=[[1.0, 0.0], [0.0, 1.0]], metadatas=[{"g": ["a", "b"]}, {"g": ["c"]}])
assert probe.get(where={"g": {"$contains": "b"}})["ids"] == ["p1"], "list metadata $contains not supported"
client.delete_collection("capability_probe")
print("list metadata + $contains on lists supported ✔ -> allowed_groups stored as a list")

pd.DataFrame([{"industry (CURRENT alias)": i, "collection": n, "space": client.get_collection(n).configuration["hnsw"]["space"],
               "count": client.get_collection(n).count()} for i, n in tables.items()])

## 6. Upload all industry documents (Word/PDF → LangChain → gates → Chroma)

In [ ]:
results, summary = {}, []
for ind in INDUSTRIES:
    lc, missing_meta = load_langchain_documents(ind)
    docs = to_source_docs(lc)
    res = run_pipeline(docs, ind, embedder)
    res.rejections += missing_meta
    save_sources(ind, res.clean_docs)
    upsert(ind, res.records)
    save_rejections(res.rejections)
    results[ind] = res
    summary.append({"industry": ind, "files": len(docs),
                    "pdf": sum(d["file"].endswith(".pdf") for d in docs), "docx": sum(d["file"].endswith(".docx") for d in docs),
                    "accepted": len(res.clean_docs), "chunks": len(res.records), "quarantined": len(res.rejections),
                    "tenants": ", ".join(sorted({d["tenant_id"] for d in docs}))})
pd.DataFrame(summary)

In [ ]:
# quarantine (fail closed): every industry's bad document stopped at a gate
pd.DataFrame(sor.execute("select industry, source_id, gate, reason from quarantine order by industry").fetchall(),
             columns=["industry", "source_id", "gate", "reason"])

In [ ]:
# stored chunks carry the industry-specific fields as typed metadata (dates as int yyyymmdd)
got = col("healthcare").get(include=["metadatas"])
cols_ = ["source_id", "source_file", "chunk_no", "section", "tenant_id", "record_type", "patient_ref_token",
         "consent_scope", "encounter_date", "allowed_groups", "valid_to", "status"]
pd.DataFrame([{c: m.get(c) for c in cols_} for m in got["metadatas"]]).sort_values(["source_id", "chunk_no"]).reset_index(drop=True)

In [ ]:
# redaction: the email inside hc-001.pdf never reaches the index or the system of record
txt = sor.execute("select text from sources where source_id = 'hc-001'").fetchone()[0]
assert "@" not in txt and "[EMAIL]" in txt
indexed = col("healthcare").get(where={"source_id": "hc-001"}, include=["documents"])["documents"]
assert indexed and not any("@" in d for d in indexed)
print(re.search(r"[^.]*\[EMAIL\][^.]*\.?", txt).group().strip())

## 7. Idempotency — same upload again, same counts

In [ ]:
before = {i: count(i) for i in INDUSTRIES}
for ind, res in results.items():
    upsert(ind, res.records)
after = {i: count(i) for i in INDUSTRIES}
assert before == after
print("idempotent ✔", after, "| embedding cache hits:", embedder.cache_hits)

## 8. Staging → promote — staged chunks are never searchable

In [ ]:
golden = json.loads((SAMPLES / "golden_queries.json").read_text())
staged_hits = {g["query"]: len(safe_search(g["industry"], g["tenant_id"], g["groups"], embedder.embed_query(g["query"])))
               for g in golden}
assert sum(staged_hits.values()) == 0
print("hits while staged (all golden queries):", sum(staged_hits.values()))
print("promoted:", {i: promote(i) for i in INDUSTRIES})
status_rows = []
for i in INDUSTRIES:
    st = pd.Series([m["status"] for m in col(i).get(include=["metadatas"])["metadatas"]]).value_counts()
    status_rows += [{"industry": i, "status": s, "chunks": int(n)} for s, n in st.items()]
pd.DataFrame(status_rows)

## 9. Safe search — golden queries per industry

In [ ]:
rows, latencies = [], []
for g in golden:
    qv = embedder.embed_query(g["query"])
    t0 = time.perf_counter()
    hits = safe_search(g["industry"], g["tenant_id"], g["groups"], qv, k=5)
    latencies.append((time.perf_counter() - t0) * 1000)
    found = [h["source_id"] for h in hits]
    rows.append({"industry": g["industry"], "tenant": g["tenant_id"], "query": g["query"], "expected": g["expected_source_id"],
                 "top1": found[0] if found else None, "top1 section": hits[0]["section"] if hits else None,
                 "score": round(hits[0]["score"], 3) if hits else None,
                 "hit@1": found[:1] == [g["expected_source_id"]], "hit@5": g["expected_source_id"] in found})
search_df = pd.DataFrame(rows)
assert search_df["hit@5"].all(), search_df[~search_df["hit@5"]]
print(f"hit@1 = {search_df['hit@1'].mean():.2f} | hit@5 = {search_df['hit@5'].mean():.2f} | p50 latency = {statistics.median(latencies):.1f} ms")
search_df

In [ ]:
print(json.dumps(safe_where("forgeworks", ["maintenance"]), indent=1))       # the exact pre-filter sent to Chroma
pd.DataFrame(safe_search("manufacturing", "forgeworks", ["maintenance"], embedder.embed_query("how to lock out the hydraulic press")))

## 10. Isolation tests — Chroma is **filter-only**

Honest statement: embedded Chroma has **no server-side tenant enforcement** (no RLS, no per-request identity, no auth in `PersistentClient`). Chroma's `tenant`/`database` namespaces exist, but they are chosen by the caller, not enforced per user, and our collections hold several tenants each. Isolation therefore holds **only** because every read goes through `safe_search`, which always injects tenant + ACL into the `where` pre-filter.
1. **Filter-only probe**: an unfiltered `get()` sees *every* tenant (proves nothing in the DB stops it); the tenant filter returns only that tenant's rows.
2. **Cross-tenant**: other tenant asks the same golden question → never gets the owner's chunks.
3. **ACL**: user without the group gets nothing. 4. **Fail closed**: no tenant/groups → `PermissionError`.

In [ ]:
probe_rows = []
for ind in INDUSTRIES:
    c = col(ind)
    row = {"industry": ind, "unfiltered get() (no enforcement)": len(c.get(include=[])["ids"])}
    for t in sorted({d["tenant_id"] for d in results[ind].clean_docs}):
        metas = c.get(where={"tenant_id": t}, include=["metadatas"])["metadatas"]
        assert all(m["tenant_id"] == t for m in metas)
        row[t] = len(metas)
    tenant_total = sum(v for k, v in row.items() if k not in ("industry", "unfiltered get() (no enforcement)"))
    assert row["unfiltered get() (no enforcement)"] == tenant_total            # raw access sees all tenants
    probe_rows.append(row)
pd.DataFrame(probe_rows).astype(object).fillna("-")

In [ ]:
iso = []
for g in golden:
    other = ({d["tenant_id"] for d in results[g["industry"]].clean_docs} - {g["tenant_id"]}).pop()
    qv = embedder.embed_query(g["query"])
    hits = safe_search(g["industry"], other, g["groups"], qv)
    leaked = [h for h in hits if h["tenant_id"] != other or h["source_id"] == g["expected_source_id"]]
    no_acl = safe_search(g["industry"], g["tenant_id"], ["group_nobody_has"], qv)
    iso.append({"industry": g["industry"], "owner": g["tenant_id"], "asked as": other,
                "hits for other tenant": len(hits), "leaked": len(leaked), "hits without ACL group": len(no_acl)})
iso_df = pd.DataFrame(iso)
assert iso_df["leaked"].eq(0).all() and iso_df["hits without ACL group"].eq(0).all()
# multi-group user: $or over $contains clauses still works
assert safe_search("healthcare", "acme_health", ["group_nobody_has", "care_team_cardiology"],
                   embedder.embed_query("chest pain troponin"))[0]["source_id"] == "hc-001"
for bad in [("", ["care_team_cardiology"]), ("acme_health", []), ("acme_health", [None]), (None, ["care_team_cardiology"])]:
    try:
        safe_search("healthcare", *bad, embedder.embed_query("x")); raise AssertionError("should raise")
    except PermissionError as e:
        print("fail closed ✔", e)
# validity window is part of the pre-filter: an expired or not-yet-valid chunk drops out of the DB query
g0 = golden[0]; qv0 = embedder.embed_query(g0["query"])
top = safe_search(g0["industry"], g0["tenant_id"], g0["groups"], qv0, k=1)[0]
c0 = col(g0["industry"]); orig = c0.get(ids=[top["chunk_id"]], include=["metadatas"])["metadatas"][0]
for patch in ({"valid_to": today_ymd() - 1}, {"valid_from": today_ymd() + 1}):
    c0.update(ids=[top["chunk_id"]], metadatas=[patch])
    assert top["chunk_id"] not in [h["chunk_id"] for h in safe_search(g0["industry"], g0["tenant_id"], g0["groups"], qv0, k=5)], patch
    c0.update(ids=[top["chunk_id"]], metadatas=[{"valid_from": orig["valid_from"], "valid_to": orig["valid_to"]}])
assert safe_search(g0["industry"], g0["tenant_id"], g0["groups"], qv0, k=1)[0]["chunk_id"] == top["chunk_id"]
print("validity window enforced in the pre-filter ✔ (expired and future chunks excluded, restored after)")
iso_df

## 11. Update — new version of `hc-002.docx` supersedes v1

In [ ]:
v1 = next(d for d in to_source_docs(load_langchain_documents("healthcare")[0]) if d["source_id"] == "hc-002")
v2 = {**v1, "source_version": "2",
      "text": v1["text"].replace("Metformin dose increased to 1000 mg twice daily", "Metformin dose increased to 1500 mg extended release once daily")}
res2 = run_pipeline([v2], "healthcare", embedder)
save_sources("healthcare", res2.clean_docs)
upsert("healthcare", res2.records)
promote("healthcare", ["hc-002"])
print("superseded chunks:", supersede("healthcare", "hc-002", keep_version="2"))
hits = safe_search("healthcare", "acme_health", ["care_team_endocrine"], embedder.embed_query("metformin dose"))
hc2 = [h for h in hits if h["source_id"] == "hc-002"]
assert hc2 and all(h["source_version"] == "2" for h in hc2)
assert any("1500 mg" in h["text"] for h in hc2)
ver = pd.DataFrame([{"version": m["source_version"], "status": m["status"]}
                    for m in col("healthcare").get(where={"source_id": "hc-002"}, include=["metadatas"])["metadatas"]])
display(ver.value_counts().rename("chunks").reset_index().sort_values("version"))
pd.DataFrame(hits)[["source_id", "source_version", "section", "score", "text"]]

## 12. Delete / consent withdrawal — purge `hc-004` and verify absence

In [ ]:
q = embedder.embed_query("ejection fraction systolic dysfunction")
before = [h["source_id"] for h in safe_search("healthcare", "beacon_clinic", ["care_team_cardiology"], q)]
purged = purge("healthcare", "hc-004")
after = [h["source_id"] for h in safe_search("healthcare", "beacon_clinic", ["care_team_cardiology"], q)]
left_index = len(col("healthcare").get(where={"source_id": "hc-004"}, include=[])["ids"])
left = sor.execute("select count(*) from sources where source_id = 'hc-004'").fetchone()[0]
assert "hc-004" in before and "hc-004" not in after and left_index == 0 and left == 0
print(f"purged {purged} chunks | before: {sorted(set(before))} | after: {sorted(set(after))} | "
      f"index rows left: {left_index} | SoR rows left: {left} ✔")

## 13. Metrics (for the cross-DB comparison notebook)

In [ ]:
metrics = {"db": "chroma", "industries": len(INDUSTRIES), "chunks_total": sum(count(i) for i in INDUSTRIES),
           "quarantined": sum(len(r.rejections) for r in results.values()),
           "hit@1": float(search_df["hit@1"].mean()), "hit@5": float(search_df["hit@5"].mean()),
           "latency_p50_ms": round(statistics.median(latencies), 2),
           "isolation": "PASS (tests) - filter-only: tenant + ACL($contains) + status/validity in Chroma where pre-filter "
                        "via safe_search; NO server-side enforcement (embedded, no RLS/auth)",
           "executed": True}
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "chroma.json").write_text(json.dumps(metrics, indent=2))
metrics

## Next: embedding model change (blue/green)
Never re-embed into the existing collections. Create `{industry}_v1_<newmodel>_<dim>` with a new `Embedder`, re-embed from `sources` in `data/sor/chroma.db` (system of record), run the golden queries against it, then repoint `CURRENT[industry]` to the new collection name (Chroma has no alias, so this dict, or a config row persisted next to it, is the switch) and delete the old collection.